# 5. ANN 实验 — 人工神经网络分类与回归

**目标数据集：** [Iris (鸢尾花)](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset) / [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)

**核心任务：** 使用人工神经网络 (ANN) 进行分类与回归，掌握网络结构设计、激活函数选择、特征预处理对模型性能的影响

**实验流程：**

| 步骤 | 内容 | 掌握技能 |
|------|------|----------|
| 1 | 环境验证 | Python / scikit-learn / matplotlib |
| 2 | ANN 简介 | 网络结构 / 前向传播 / 反向传播 / 激活函数 |
| 3 | ANN 分类 + 交叉验证 | MLPClassifier / cross_val_score / ROC 曲线 |
| 4 | 特征选择与提取 | Pipeline / SelectKBest / PCA |
| 5 | 激活函数对比 | logistic vs relu vs tanh |
| 6 | ANN 回归 | MLPRegressor / 加州房价 / 损失曲线 |
| 7 | ANN 探索* | 特征选择 / PCA / 不同激活函数 / solver 对比 |

> 注：标星号(*)的节为进阶探索内容。

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"numpy: {np.__version__}")

import sklearn
print(f"scikit-learn: {sklearn.__version__}")

import matplotlib
print(f"matplotlib: {matplotlib.__version__}")

import pandas as pd
print(f"pandas: {pd.__version__}")

import seaborn as sns
print(f"seaborn: {sns.__version__}")

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_curve, auc, mean_squared_error
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 定义配色方案（与 02/03/04 保持一致）
COLORS = ["#e74c3c", "#2ecc71", "#3498db"]
LABELS = ['setosa', 'versicolor', 'virginica']

# 加载数据
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"\nIris 数据集形状: {X_iris.shape}")
print(f"类别: {LABELS}")

print("\n环境就绪 ✓")

---
## 2. ANN 简介

人工神经网络 (Artificial Neural Network, ANN) 是一种模拟生物神经系统的机器学习模型。本实验使用 scikit-learn 的 MLPClassifier 和 MLPRegressor 实现多层感知机（Multi-Layer Perceptron, MLP）。

### 2.1 网络结构

MLP 由输入层、隐藏层和输出层组成：

```
多层感知机结构
────────────────

输入层        隐藏层 1      隐藏层 2      输出层
(4 features)  (10 neurons)  (5 neurons)  (3 classes)

  x1 ──────→  h1 ────────→  h1' ───────→  y1
  x2 ──────→  h2 ────────→  h2' ───────→  y2
  x3 ──────→  h3 ────────→  h3' ───────→  y3
  x4 ──────→  ...         →  ...        →
              h10 ───────→  h5' ────────→

每层神经元通过权重连接，数据从左向右流动（前向传播）
```

### 2.2 前向传播与反向传播

**前向传播**：$h = \sigma(W \cdot x + b)$

**反向传播**：根据输出误差，从输出层向输入层反向计算梯度并更新参数。

**优化器 (Solver)**：

| Solver | 说明 | 适用场景 |
|--------|------|----------|
| adam | 自适应矩估计 | 大数据集，默认选择 |
| sgd | 随机梯度下降 | 需精细调参 |
| lbfgs | 牛顿法近似 | 小数据集 |

### 2.3 激活函数

| 激活函数 | 公式 | 特点 |
|----------|------|------|
| relu | $f(x) = \max(0, x)$ | 计算快，避免梯度消失，最常用 |
| logistic (sigmoid) | $f(x) = \frac{1}{1+e^{-x}}$ | 输出 0-1，适合二分类输出层 |
| tanh | $f(x) = \frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 输出 -1 到 1，中心化 |
| identity | $f(x) = x$ | 无变换，用于回归输出层 |

---
## 3. ANN 分类 + 交叉验证

本节使用 MLPClassifier 对 Iris 数据集进行分类。

### 3.1 MLPClassifier 关键参数

| 参数 | 说明 | 示例 |
|------|------|------|
| hidden_layer_sizes | 隐藏层结构 | (10, 5) 表示两个隐藏层 |
| activation | 激活函数 | 'relu', 'logistic', 'tanh' |
| solver | 优化器 | 'adam', 'sgd', 'lbfgs' |
| max_iter | 最大迭代次数 | 1000 |

### 3.2 数据预处理

神经网络对特征尺度敏感，通常需要标准化处理：

$$z = \frac{x - \mu}{\sigma}$$

标准化有助于：
- 加速收敛
- 避免某些特征因数值大而主导训练
- 使梯度下降更稳定

In [ ]:
# 划分训练集和测试集
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

# 标准化
scaler_i = StandardScaler()
X_train_i_scaled = scaler_i.fit_transform(X_train_i)
X_test_i_scaled = scaler_i.transform(X_test_i)

print(f"训练集样本数: {len(X_train_i)}")
print(f"测试集样本数: {len(X_test_i)}")

### 3.3 模型训练与交叉验证

使用 cross_val_score 进行 5-fold 交叉验证，评估模型在不同数据划分上的性能稳定性。

In [ ]:
# 创建多层感知机分类器
mlp_iris = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    solver='adam',
    max_iter=1000,
    random_state=42
)

# 交叉验证评估模型
scores = cross_val_score(mlp_iris, X_train_i_scaled, y_train_i, cv=5)
print(f"交叉验证准确率: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"各折得分: {scores}")

# 训练模型
mlp_iris.fit(X_train_i_scaled, y_train_i)

# 在测试集上评估
test_accuracy = mlp_iris.score(X_test_i_scaled, y_test_i)
print(f"\n测试集准确率: {test_accuracy:.4f}")

### 3.4 ROC 曲线

ROC 曲线（Receiver Operating Characteristic Curve）是评估分类模型性能的重要工具。对于多分类问题，可以为每个类别绘制一条 ROC 曲线。

**ROC 曲线解读：**

| 概念 | 含义 |
|------|------|
| TPR (True Positive Rate) | 真阳性率 = 正确预测为正 / 实际为正 |
| FPR (False Positive Rate) | 假阳性率 = 错误预测为正 / 实际为负 |
| AUC (Area Under Curve) | ROC 曲线下面积，越大越好 |

In [ ]:
# 计算 ROC 曲线
y_score = mlp_iris.predict_proba(X_test_i_scaled)

# 为每个类别绘制 ROC 曲线
fpr = dict()
tpr = dict()
roc_auc = dict()

plt.figure(figsize=(10, 8))

for i in range(len(LABELS)):
    fpr[i], tpr[i], _ = roc_curve((y_test_i == i).astype(int), y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], 
             color=COLORS[i],
             label=f'{LABELS[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - MLPClassifier on Iris')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. 特征选择与提取

本节对比特征选择（SelectKBest）和特征提取（PCA）对 ANN 分类性能的影响。

### 4.1 方法对比

| 方法 | 说明 | 优点 |
|------|------|------|
| SelectKBest | 选择最重要的 k 个特征 | 保留可解释性 |
| PCA | 降维到 k 个主成分 | 保留最大方差 |

In [ ]:
# 特征选择：选择最佳的 2 个特征
selector = SelectKBest(f_classif, k=2)
X_iris_selected = selector.fit_transform(X_iris, y_iris)
print(f"选择后的特征数: {X_iris_selected.shape[1]}")

# 特征提取：PCA 降维到 2 个主成分
pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris)
print(f"PCA 后的特征数: {X_iris_pca.shape[1]}")
print(f"解释方差比: {pca.explained_variance_ratio_}")

In [ ]:
# 对比三种方案的性能
results = []

# 全部特征
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp_full = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
mlp_full.fit(X_train_scaled, y_train)
acc_full = mlp_full.score(X_test_scaled, y_test)
results.append(('全部特征 (4D)', acc_full))

# 特征选择
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_iris_selected, y_iris, test_size=0.3, random_state=42)
scaler_s = StandardScaler()
X_train_s_scaled = scaler_s.fit_transform(X_train_s)
X_test_s_scaled = scaler_s.transform(X_test_s)

mlp_selected = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
mlp_selected.fit(X_train_s_scaled, y_train_s)
acc_selected = mlp_selected.score(X_test_s_scaled, y_test_s)
results.append(('Feature Selection (2D)', acc_selected))

# PCA
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_iris_pca, y_iris, test_size=0.3, random_state=42)
scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

mlp_pca = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
mlp_pca.fit(X_train_p_scaled, y_train_p)
acc_pca = mlp_pca.score(X_test_p_scaled, y_test_p)
results.append(('PCA (2D)', acc_pca))

print("性能对比:")
for name, acc in results:
    print(f"  {name}: {acc:.4f}")

---
## 5. 激活函数对比

不同激活函数对神经网络的性能有重要影响。

### 5.1 常用激活函数

| 激活函数 | 公式 | 特点 |
|----------|------|------|
| relu | $f(x) = \max(0, x)$ | 计算快，避免梯度消失 |
| logistic | $f(x) = \frac{1}{1+e^{-x}}$ | 输出 0-1 |
| tanh | $f(x) = \frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 输出 -1 到 1 |

In [ ]:
# 创建使用不同激活函数的 MLP 分类器
mlp_sigmoid = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    activation='logistic',
    max_iter=1000,
    random_state=42
)
mlp_relu = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    activation='relu',
    max_iter=1000,
    random_state=42
)
mlp_tanh = MLPClassifier(
    hidden_layer_sizes=(10, 5),
    activation='tanh',
    max_iter=1000,
    random_state=42
)

# 训练并评估
mlp_sigmoid.fit(X_train_scaled, y_train)
mlp_relu.fit(X_train_scaled, y_train)
mlp_tanh.fit(X_train_scaled, y_train)

score_sigmoid = mlp_sigmoid.score(X_test_scaled, y_test)
score_relu = mlp_relu.score(X_test_scaled, y_test)
score_tanh = mlp_tanh.score(X_test_scaled, y_test)

print(f"Sigmoid 激活函数准确率: {score_sigmoid:.4f}")
print(f"ReLU 激活函数准确率: {score_relu:.4f}")
print(f"Tanh 激活函数准确率: {score_tanh:.4f}")

---
## 6. ANN 回归

本节使用 MLPRegressor 对 California Housing 数据集进行回归预测。

### 6.1 MLPRegressor 关键参数

| 参数 | 说明 |
|------|------|
| hidden_layer_sizes | 隐藏层结构 |
| activation | 激活函数 |
| solver | 优化器 (adam/sgd/lbfgs) |
| max_iter | 最大迭代次数 |

### 6.2 数据加载与预处理

In [ ]:
# 加载加州房价数据集
housing = fetch_california_housing()
X_house, y_house = housing.data, housing.target

print(f"样本数: {X_house.shape[0]}, 特征数: {X_house.shape[1]}")
print(f"特征: {housing.feature_names}")

### 6.3 数据划分与标准化

In [ ]:
# 划分训练集和测试集
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.3, random_state=42
)

# 标准化（对神经网络非常重要）
scaler_h = StandardScaler()
X_train_h_scaled = scaler_h.fit_transform(X_train_h)
X_test_h_scaled = scaler_h.transform(X_test_h)

print(f"训练集样本数: {len(X_train_h)}")
print(f"测试集样本数: {len(X_test_h)}")

### 6.4 模型训练

In [ ]:
# 创建并训练 MLP 回归模型
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(10, 5),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

mlp_reg.fit(X_train_h_scaled, y_train_h)

# 预测
y_pred_h = mlp_reg.predict(X_test_h_scaled)

# 评估
mse = mean_squared_error(y_test_h, y_pred_h)
r2 = mlp_reg.score(X_test_h_scaled, y_test_h)

print(f"均方误差 (MSE): {mse:.4f}")
print(f"决定系数 (R²): {r2:.4f}")

### 6.5 损失曲线可视化

In [ ]:
# 绘制损失曲线
plt.figure(figsize=(10, 6))
plt.plot(mlp_reg.loss_curve_)
plt.title('MLPRegressor Training Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.show()

### 6.6 预测结果可视化

In [ ]:
# 预测结果可视化
plt.figure(figsize=(10, 6))
plt.scatter(y_test_h, y_pred_h, alpha=0.5)
plt.plot([y_house.min(), y_house.max()], [y_house.min(), y_house.max()], 'r--', label='理想预测 (y=x)')
plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.title('MLPRegressor: Prediction vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. ANN 探索*

本节探索不同方法对 ANN 回归性能的影响，包括特征选择、PCA 降维以及不同激活函数的对比。

### 7.1 探索方向

| 探索方向 | 方法 |
|----------|------|
| 特征选择 | SelectKBest |
| 特征提取 | PCA 降维 |
| 激活函数 | relu vs logistic |
| 优化器 | adam vs sgd |

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

# 特征选择：选择最佳的 5 个特征
selector_h = SelectKBest(f_regression, k=5)
X_house_selected = selector_h.fit_transform(X_house, y_house)
print(f"选择后的特征数: {X_house_selected.shape[1]}")

In [ ]:
# PCA 降维到 5 个主成分
pca_h = PCA(n_components=5)
X_house_pca = pca_h.fit_transform(X_house)
print(f"PCA 后的特征数: {X_house_pca.shape[1]}")
print(f"解释方差比: {pca_h.explained_variance_ratio_}")
print(f"累计解释方差: {pca_h.explained_variance_ratio_.sum():.4f}")

In [ ]:
# 对比不同配置的性能
results_explore = []

# 全部特征 + relu
mlp_base = MLPRegressor(hidden_layer_sizes=(10, 5), max_iter=500, random_state=42)
mlp_base.fit(X_train_h_scaled, y_train_h)
mse_base = mean_squared_error(y_test_h, mlp_base.predict(X_test_h_scaled))
results_explore.append(('全部特征 + relu', mse_base))

# 特征选择
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_house_selected, y_house, test_size=0.3, random_state=42
)
scaler_s = StandardScaler()
X_train_s_scaled = scaler_s.fit_transform(X_train_s)
X_test_s_scaled = scaler_s.transform(X_test_s)

mlp_selected = MLPRegressor(hidden_layer_sizes=(10, 5), max_iter=500, random_state=42)
mlp_selected.fit(X_train_s_scaled, y_train_s)
mse_selected = mean_squared_error(y_test_s, mlp_selected.predict(X_test_s_scaled))
results_explore.append(('特征选择 (5D)', mse_selected))

# PCA
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_house_pca, y_house, test_size=0.3, random_state=42
)
scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

mlp_pca = MLPRegressor(hidden_layer_sizes=(10, 5), max_iter=500, random_state=42)
mlp_pca.fit(X_train_p_scaled, y_train_p)
mse_pca = mean_squared_error(y_test_p, mlp_pca.predict(X_test_p_scaled))
results_explore.append(('PCA (5D)', mse_pca))

print("回归性能对比 (MSE 越小越好):")
for name, mse in results_explore:
    print(f"  {name}: {mse:.4f}")

In [ ]:
# 激活函数对比
mlp_sigmoid = MLPRegressor(
    hidden_layer_sizes=(10, 5),
    activation='logistic',
    max_iter=500,
    random_state=42
)
mlp_sigmoid.fit(X_train_h_scaled, y_train_h)
mse_sigmoid = mean_squared_error(y_test_h, mlp_sigmoid.predict(X_test_h_scaled))

mlp_relu = MLPRegressor(
    hidden_layer_sizes=(10, 5),
    activation='relu',
    max_iter=500,
    random_state=42
)
mlp_relu.fit(X_train_h_scaled, y_train_h)
mse_relu = mean_squared_error(y_test_h, mlp_relu.predict(X_test_h_scaled))

print(f"Sigmoid 激活函数 MSE: {mse_sigmoid:.4f}")
print(f"ReLU 激活函数 MSE: {mse_relu:.4f}")

---
## 总结

### 方法对比总表

| 方法 | 类型 | 核心思想 | 优点 | 缺点 |
|------|------|----------|------|------|
| MLPClassifier | 分类 | 多层感知机 + 交叉熵损失 | 非线性建模能力强 | 需要调参，易过拟合 |
| MLPRegressor | 回归 | 多层感知机 + 均方误差损失 | 适合复杂回归问题 | 训练时间较长 |
| adam | 优化器 | 自适应矩估计 | 收敛快，默认选择 | 可能陷入局部最优 |
| sgd | 优化器 | 随机梯度下降 | 简单直观 | 需要调学习率 |

### 核心知识点

| 知识点 | 要点 |
|--------|------|
| 网络结构 | 输入层 → 隐藏层 → 输出层 |
| 前向传播 | $h = \sigma(W \cdot x + b)$ |
| 反向传播 | 根据误差更新权重和偏置 |
| 激活函数 | ReLU 最常用，避免梯度消失 |
| 数据标准化 | 神经网络必需，加速收敛 |
| 损失曲线 | 监控训练过程，判断是否收敛 |

### 思考题

1. 隐藏层节点数如何影响模型性能？越多越好吗？
2. 什么时候选择 sgd 而不是 adam？
3. 如何判断神经网络是否过拟合？
4. 特征选择和特征提取对神经网络性能的影响有何不同？